In [15]:
import pandas as pd
import numpy as np
import re
#import nb_black

johor = pd.read_csv("johor_prop_20250418.csv")
johor.head()

# show full row
pd.options.display.max_columns = None

In [16]:
# remove unwanted column

johor = johor[['House Type', 'Price', 'Location', 'Size (sq.ft)',
       'No.of Bed', 'No.of Bath', 'Land Status']]
print(johor.head())
print(johor.shape)

                                          House Type       Price  \
0       New Condominium  for sale, listed 3 days ago  RM 190,000   
1  New 2-storey Terraced House  for sale, listed ...  RM 849,000   
2      New Cluster House  for sale, listed 1 day ago  RM 999,000   
3  New Service Residence  for sale, listed 2 hour...  RM 330,000   
4        New Apartment  for sale, listed 4 hours ago  RM 300,000   

                        Location Size (sq.ft)  No.of Bed  No.of Bath  \
0             Johor Bahru, Johor          900        3.0         3.0   
1                  Skudai, Johor        1,400        4.0         4.0   
2                  Skudai, Johor        2,240        4.0         4.0   
3  D' Secret Garden, Johor Bahru          667        2.0         2.0   
4    Austin Perdana, Johor Bahru          495        1.0         1.0   

  Land Status  
0    Freehold  
1    Freehold  
2    Freehold  
3    Freehold  
4    Freehold  
(323, 7)


In [17]:
# renaming columns name for standardization

johor.rename(columns= {
    'House Type': 'property_type',
    'Price': 'price', 
    'Location': 'location', 
    'Size (sq.ft)': 'size_sqft', 
    'No.of Bed': 'total_bedroom',
    'No.of Bath': 'total_bathroom', 
    'Land Status': 'land_status'
}, inplace = True)

johor.columns


Index(['property_type', 'price', 'location', 'size_sqft', 'total_bedroom',
       'total_bathroom', 'land_status'],
      dtype='object')

In [18]:
# cleaning each column before proceed to Statistical Analysis

# 1. property_type

johor["property_type"].head()

house_types = []

regex_property = r"(?<=New\s)(\S+)"


for house in johor["property_type"]:
    house_matches = re.search(regex_property, house)
    if house_matches is not None:
        house_type = house_matches.group(0).strip()
    else:
        house_type = ""
    house_types.append(house_type)
    
#print(house_types)

johor["property_type_extract"] = house_types
print(johor.head())

                                       property_type       price  \
0       New Condominium  for sale, listed 3 days ago  RM 190,000   
1  New 2-storey Terraced House  for sale, listed ...  RM 849,000   
2      New Cluster House  for sale, listed 1 day ago  RM 999,000   
3  New Service Residence  for sale, listed 2 hour...  RM 330,000   
4        New Apartment  for sale, listed 4 hours ago  RM 300,000   

                        location size_sqft  total_bedroom  total_bathroom  \
0             Johor Bahru, Johor       900            3.0             3.0   
1                  Skudai, Johor     1,400            4.0             4.0   
2                  Skudai, Johor     2,240            4.0             4.0   
3  D' Secret Garden, Johor Bahru       667            2.0             2.0   
4    Austin Perdana, Johor Bahru       495            1.0             1.0   

  land_status property_type_extract  
0    Freehold           Condominium  
1    Freehold              2-storey  
2    Freehold 

In [19]:
# 2. price

johor["prices"] = johor["price"].str.split(expand=True)[1].str.replace(",","")
johor["prices"] = johor["prices"].astype("int")
johor["prices"].dtype

dtype('int32')

In [20]:
# 3. location - since all properties in the dataset are from Johor, so we wil focus on the district only

# lower case each letter
johor["location"] = [x.lower() for x in johor["location"]]
# Check for no. of district under Johor < suppose total under 
#print(johor["location"].value_counts())  # total 58

# check the value of the unknown district
#print(johor["location"].nunique())

# Handling inconsistent lcoation names

# info from google - Batu Pahat, Johor Bahru, Kluang, Kota Tinggi, Mersing, Muar, Pontian, and Segamat (8 district)

valid_districts = {
    "batu pahat": "batu pahat",
    "johor bahru": "johor bahru",
    "kluang": "kluang",
    "kota tinggi": "kota tinggi",
    "mersing": "mersing",
    "muar": "muar",
    "pontian": "pontian",
    "segamat": "segamat"
}

town_to_district = {
    "johor bahru": "johor bahru",
    "kulai": "kulai",
    "kluang": "kluang",
    "pasir gudang": "johor bahru",
    "senai": "kulai",
    "skudai": "johor bahru",
    "iskandar puteri": "johor bahru",
    "permas jaya": "johor bahru",
    "masai": "johor bahru",
    "muar": "johor",
    "gelang patah": "johor bahru",
    "parkland by the river": "johor bahru",
    "tebrau": "johor bahru",
    "sentrio residences @ senai": "kulai",
    "horizon hills": "johor bahru",
    "kota tinggi": "kota tinggi",
    "pangsapuri ksl bukit gemilang": "johor bahru",
    "tangkak": "muar",
    "ayer hitam": "kluang",
    "d' secret garden": "johor bahru",
    "pengerang": "kota tinggi",
    "bandar baru permas jaya": "johor bahru",
    "parc regency": "johor bahru",
    "ksl residence 2 @ kangkar tebrau": "johor bahru",
    "veranda residence": "johor bahru",
    "setia indah": "johor bahru",
    "d'secret garden @ kempas indah": "johor bahru",
    "verte medini condominium": "johor bahru",
    "desaru utama residence": "kota tinggi",
    "the senai garden": "kulai",
    "batu pahat": "batu pahat",
    "pontian": "pontian",
    "pandan residence": "johor bahru",
    "bakri": "muar",
    "mersing": "mersing",
    "sierra heights (residensi siera perdana)": "johor bahru",
    "yong peng": "kluang",
    "puteri harbour": "johor bahru",
    "santai @ eco spring": "johor bahru",
    "mutiara austin": "johor bahru",
    "east bay (seri bayan)": "johor bahru",
    "the garden residences": "johor bahru",
    "senibong": "johor bahru",
    "m minori": "johor bahru",
    "d'summit residences": "johor bahru",
    "seri austin residence luxury apartment": "johor bahru",
    "marina cove": "johor bahru",
    "setia tropika": "johor bahru",
    "permas sentral": "johor bahru",
    "idaman residence @ nusa idaman": "johor bahru",
    "tampoi height serviced apartment": "johor bahru",
    "iskandar residences medini": "johor bahru",
    "vista tiara @ mbw bay": "johor bahru",
    "arc @ austin hills": "johor bahru",
    "ksl residences @ daya": "johor bahru",
    "aliva @ mount austin": "johor bahru",
    "kings bay @ country garden danga bay": "johor bahru",
    "ulu tiram": "johor bahru"
}

# function to get the correct district

def get_district(location):
    parts = location.split(",")
    for part in parts:
        part = part.strip()
        if part in valid_districts:
            return valid_districts[part]
        elif part in town_to_district:
            return town_to_district[part]
    else:
        return "Unknown"
        
johor["district"] = johor["location"].apply(get_district)
print(johor["district"].value_counts())

district
johor bahru    211
kulai           43
kluang          41
muar            11
kota tinggi     10
batu pahat       3
Unknown          2
mersing          1
pontian          1
Name: count, dtype: int64


In [21]:
johor.head()

,property_type,price,location,size_sqft,total_bedroom,total_bathroom,land_status,property_type_extract,prices,district
0,"New Condominium for sale, listed 3 days ago","RM 190,000","johor bahru, johor",900,3.0,3.0,Freehold,Condominium,190000,johor bahru
1,"New 2-storey Terraced House for sale, listed ...","RM 849,000","skudai, johor","1,400",4.0,4.0,Freehold,2-storey,849000,johor bahru
2,"New Cluster House for sale, listed 1 day ago","RM 999,000","skudai, johor","2,240",4.0,4.0,Freehold,Cluster,999000,johor bahru
3,"New Service Residence for sale, listed 2 hour...","RM 330,000","d' secret garden, johor bahru",667,2.0,2.0,Freehold,Service,330000,johor bahru
4,"New Apartment for sale, listed 4 hours ago","RM 300,000","austin perdana, johor bahru",495,1.0,1.0,Freehold,Apartment,300000,johor bahru


In [22]:
# size_sqft

johor["size_sqft"] = johor["size_sqft"].str.replace(",", "").astype("int")
johor["size_sqft"].dtype



dtype('int32')

In [23]:
# property_type_extract

valid_property_type = {
    "landed": "landed",
    "high_rise": "high_rise",
    "shop_lot": "shop_lot",
    "factory": "factory"
}

property_type = {
    "Service": "high_rise",
    "2-storey": "landed",
    "1-storey": "landed",
    "Apartment": "high_rise",
    "Terraced": "landed",
    "Semi-Detached": "landed",
    "Condominium": "high_rise",
    "Cluster": "landed",
    "Shop": "shop_lot",
    "Others": "shop_lot",
    "Warehouse": "factory",
    "Bungalow": "landed",
    "3-storey": "high_rise",
    "2.5-storey": "high_rise",
    "Office": "shop_lot",
    "Flat": "high_rise"
}

def get_type(property_name):
    return property_type.get(property_name, "Unknown")

johor["property_type_extract"] = johor["property_type_extract"].apply(get_type)
johor["property_type_extract"].value_counts()

property_type_extract
landed       165
high_rise    141
shop_lot      10
factory        6
Unknown        1
Name: count, dtype: int64

In [24]:

# remove unwanted columns
johor = johor[["district", "property_type_extract", "size_sqft", "total_bedroom", "total_bathroom", "land_status", "prices"]]
johor.rename(columns={
    "property_type_extract": "property_type",
    "district": "location"}, inplace=True)
johor.head()

,location,property_type,size_sqft,total_bedroom,total_bathroom,land_status,prices
0,johor bahru,high_rise,900,3.0,3.0,Freehold,190000
1,johor bahru,landed,1400,4.0,4.0,Freehold,849000
2,johor bahru,landed,2240,4.0,4.0,Freehold,999000
3,johor bahru,high_rise,667,2.0,2.0,Freehold,330000
4,johor bahru,high_rise,495,1.0,1.0,Freehold,300000


In [25]:
# Handling duplicated values

johor = johor.drop_duplicates()
johor.duplicated().sum()


0

In [26]:
# Handling null values


johor.dropna(subset=["total_bedroom", "total_bathroom", "land_status"], inplace=True)
johor = johor.reset_index(drop=True)
print(johor.isna().sum())

location          0
property_type     0
size_sqft         0
total_bedroom     0
total_bathroom    0
land_status       0
prices            0
dtype: int64


In [27]:
johor.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\Project 8 - Johor Property\transform_johor_prop.csv", index=False)